# 03 - Cleaning, Wrangling, and Feature Engineering

This notebook turns the raw NBA API outputs into a modeling-ready team-game dataset for notebook 04.

The central rule is simple: every feature must be knowable before tip-off. The EDA showed that same-game `PLUS_MINUS`, points, shooting, rebounds, assists, turnovers, steals, and blocks explain wins very strongly, but using them directly would leak the answer. Here we convert those basketball signals into shifted rolling, season-to-date, prior-season, opponent, rest, and matchup features.

Important design choices from the EDA and project proposal:

- Keep one row per team per game, with `is_win` as the binary target.
- Keep home/away and neutral-site flags because home court matters and is known before the game.
- Use shifted rolling windows within each team-season so the current game's box score is never included in its own predictors.
- Treat `NET_RATING` carefully: use prior-season values and shifted/to-date rolling estimates, not same-game or final-current-season values.
- Save both a broad feature table and a stricter model-ready table for time-aware validation in notebook 04.

## 1. Setup

Load packages, set paths, and define the raw and processed directories.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 160)
pd.set_option('display.float_format', '{:,.4f}'.format)

RAW_DIR = Path('data/raw')
GAME_LOG_DIR = RAW_DIR / 'game_logs'
TEAM_STATS_DIR = RAW_DIR / 'team_stats'
PROCESSED_DIR = Path('data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_PATH = PROCESSED_DIR / 'team_game_features.csv'
MODEL_READY_PATH = PROCESSED_DIR / 'modeling_dataset.csv'
DICTIONARY_PATH = PROCESSED_DIR / 'feature_dictionary.csv'

print(f'Raw game logs: {GAME_LOG_DIR}')
print(f'Raw team stats: {TEAM_STATS_DIR}')
print(f'Processed output: {PROCESSED_DIR}')

Raw game logs: data/raw/game_logs
Raw team stats: data/raw/team_stats
Processed output: data/processed


## 2. Load Raw Data

The acquisition notebook saved one CSV per season. We load all seasons with stable string IDs so joins do not lose leading zeros.

In [2]:
game_log_files = sorted(GAME_LOG_DIR.glob('game_logs_*.csv'))
team_stats_files = sorted(TEAM_STATS_DIR.glob('team_stats_*.csv'))

assert game_log_files, 'No game log CSVs found under data/raw/game_logs/.'
assert team_stats_files, 'No team stats CSVs found under data/raw/team_stats/.'

game_logs_all = pd.concat(
    [pd.read_csv(path, dtype={'GAME_ID': 'string', 'SEASON_YEAR': 'string', 'SEASON': 'string'}) for path in game_log_files],
    ignore_index=True,
)
team_stats_all = pd.concat(
    [pd.read_csv(path, dtype={'SEASON': 'string'}) for path in team_stats_files],
    ignore_index=True,
)

load_summary = pd.DataFrame(
    [
        {'dataset': 'game_logs_all', 'files': len(game_log_files), 'rows': game_logs_all.shape[0], 'columns': game_logs_all.shape[1]},
        {'dataset': 'team_stats_all', 'files': len(team_stats_files), 'rows': team_stats_all.shape[0], 'columns': team_stats_all.shape[1]},
    ]
)

load_summary

,dataset,files,rows,columns
0,game_logs_all,25,60048,58
1,team_stats_all,25,746,47


## 3. Basic Cleaning and Leakage Guardrails

Drop endpoint metadata columns, standardize dates and target fields, parse home/away status, and flag neutral or unresolved games. Current-game box-score columns remain temporarily only so we can create lagged historical features.

In [3]:
rank_columns = [column for column in game_logs_all.columns if column.endswith('_RANK')]
metadata_columns_to_drop = ['AVAILABLE_FLAG'] + rank_columns

season_order_lookup = pd.Series(
    data=range(1, game_logs_all['SEASON_YEAR'].nunique() + 1),
    index=sorted(game_logs_all['SEASON_YEAR'].dropna().unique()),
)

game_logs_clean = (
    game_logs_all
    .drop(columns=metadata_columns_to_drop, errors='ignore')
    .assign(
        GAME_DATE=lambda frame: pd.to_datetime(frame['GAME_DATE'], errors='raise').dt.normalize(),
        is_win=lambda frame: frame['WL'].map({'W': 1, 'L': 0}).astype('int8'),
        is_home=lambda frame: frame['MATCHUP'].str.contains(' vs. ', regex=False).astype('int8'),
        opponent_abbr=lambda frame: frame['MATCHUP'].str.extract(r'(?:vs\.|@)\s+([A-Z]{2,3})', expand=False),
        season_start_year=lambda frame: frame['SEASON_YEAR'].str.slice(0, 4).astype('int16'),
        game_month=lambda frame: frame['GAME_DATE'].dt.month.astype('int8'),
        game_year=lambda frame: frame['GAME_DATE'].dt.year.astype('int16'),
        season_order=lambda frame: frame['SEASON_YEAR'].map(season_order_lookup).astype('int16'),
    )
)

home_counts_by_game = game_logs_clean.groupby('GAME_ID')['is_home'].sum()
neutral_or_unresolved_game_ids = home_counts_by_game.loc[home_counts_by_game.ne(1)].index

game_logs_clean = game_logs_clean.assign(
    is_neutral_or_unresolved=lambda frame: frame['GAME_ID'].isin(neutral_or_unresolved_game_ids).astype('int8')
)

team_rank_columns = [column for column in team_stats_all.columns if column.endswith('_RANK')]
team_stats_clean = (
    team_stats_all
    .drop(columns=team_rank_columns, errors='ignore')
    .assign(
        season_start_year=lambda frame: frame['SEASON'].str.slice(0, 4).astype('int16'),
        season_order=lambda frame: frame['SEASON'].map(season_order_lookup).astype('Int16'),
    )
)

assert not game_logs_clean.duplicated(['GAME_ID', 'TEAM_ID']).any()
assert game_logs_clean.groupby('GAME_ID').size().eq(2).all()
assert set(game_logs_clean['WL'].unique()) == {'W', 'L'}

pd.DataFrame(
    [
        {'check': 'team-game rows', 'value': len(game_logs_clean)},
        {'check': 'unique games', 'value': game_logs_clean['GAME_ID'].nunique()},
        {'check': 'neutral or unresolved games', 'value': int(len(neutral_or_unresolved_game_ids))},
        {'check': 'dropped endpoint metadata columns', 'value': len(metadata_columns_to_drop)},
    ]
)

,check,value
0,team-game rows,60048
1,unique games,30024
2,neutral or unresolved games,5
3,dropped endpoint metadata columns,27


## 4. Build Opponent Rows

Each game has two team rows. Joining each row to the other row gives us the opponent ID and historical defensive outcomes like points allowed. These same-game opponent stats are used only as past observations inside shifted rolling features.

In [4]:
opponent_source_columns = [
    'GAME_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'PTS', 'FGM', 'FGA', 'FG_PCT',
    'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST',
    'TOV', 'STL', 'BLK', 'PF', 'PLUS_MINUS', 'is_win', 'is_home'
]

opponent_rows = game_logs_clean[opponent_source_columns].rename(
    columns={
        'TEAM_ID': 'opponent_team_id',
        'TEAM_ABBREVIATION': 'opponent_abbreviation_from_row',
        'TEAM_NAME': 'opponent_team_name',
        'PTS': 'opp_PTS',
        'FGM': 'opp_FGM',
        'FGA': 'opp_FGA',
        'FG_PCT': 'opp_FG_PCT',
        'FG3M': 'opp_FG3M',
        'FG3A': 'opp_FG3A',
        'FG3_PCT': 'opp_FG3_PCT',
        'FTM': 'opp_FTM',
        'FTA': 'opp_FTA',
        'FT_PCT': 'opp_FT_PCT',
        'OREB': 'opp_OREB',
        'DREB': 'opp_DREB',
        'REB': 'opp_REB',
        'AST': 'opp_AST',
        'TOV': 'opp_TOV',
        'STL': 'opp_STL',
        'BLK': 'opp_BLK',
        'PF': 'opp_PF',
        'PLUS_MINUS': 'opp_PLUS_MINUS',
        'is_win': 'opp_is_win',
        'is_home': 'opp_is_home',
    }
)

paired_games = game_logs_clean.merge(opponent_rows, on='GAME_ID', how='inner')
paired_games = paired_games.loc[paired_games['TEAM_ID'] != paired_games['opponent_team_id']].copy()

paired_games['opponent_team_id'] = paired_games['opponent_team_id'].astype('int64')
paired_games['is_road'] = ((paired_games['is_home'] == 0) & (paired_games['is_neutral_or_unresolved'] == 0)).astype('int8')

assert paired_games.shape[0] == game_logs_clean.shape[0]
assert (paired_games['is_win'] + paired_games['opp_is_win']).eq(1).all()
assert np.allclose(paired_games['PLUS_MINUS'] + paired_games['opp_PLUS_MINUS'], 0)

paired_games[['SEASON_YEAR', 'GAME_DATE', 'TEAM_ABBREVIATION', 'MATCHUP', 'WL', 'is_home', 'is_neutral_or_unresolved', 'opponent_team_id', 'opponent_team_name']].head()

,SEASON_YEAR,GAME_DATE,TEAM_ABBREVIATION,MATCHUP,WL,is_home,is_neutral_or_unresolved,opponent_team_id,opponent_team_name
1,2000-01,2001-04-18,SEA,SEA vs. SAS,W,1,0,1610612759,San Antonio Spurs
3,2000-01,2001-04-18,DAL,DAL vs. MIN,W,1,0,1610612750,Minnesota Timberwolves
5,2000-01,2001-04-18,VAN,VAN @ GSW,W,0,0,1610612744,Golden State Warriors
7,2000-01,2001-04-18,MIA,MIA @ ORL,W,0,0,1610612753,Orlando Magic
9,2000-01,2001-04-18,DEN,DEN vs. SAC,W,1,0,1610612758,Sacramento Kings


## 5. Same-Game Historical Signals

These columns describe a completed game, so they are not final model inputs by themselves. They are safe only after we shift them before computing rolling and season-to-date features.

In [5]:
feature_base = paired_games.copy()

feature_base['point_margin'] = feature_base['PLUS_MINUS'].astype(float)
feature_base['pts_allowed'] = feature_base['opp_PTS'].astype(float)
feature_base['possessions_est'] = feature_base['FGA'] + 0.44 * feature_base['FTA'] - feature_base['OREB'] + feature_base['TOV']
feature_base['opp_possessions_est'] = feature_base['opp_FGA'] + 0.44 * feature_base['opp_FTA'] - feature_base['opp_OREB'] + feature_base['opp_TOV']
feature_base['off_rating_est'] = np.where(feature_base['possessions_est'] > 0, 100 * feature_base['PTS'] / feature_base['possessions_est'], np.nan)
feature_base['def_rating_est'] = np.where(feature_base['opp_possessions_est'] > 0, 100 * feature_base['opp_PTS'] / feature_base['opp_possessions_est'], np.nan)
feature_base['net_rating_est'] = feature_base['off_rating_est'] - feature_base['def_rating_est']
feature_base['efg_pct'] = np.where(feature_base['FGA'] > 0, (feature_base['FGM'] + 0.5 * feature_base['FG3M']) / feature_base['FGA'], np.nan)
feature_base['ts_pct'] = np.where((feature_base['FGA'] + 0.44 * feature_base['FTA']) > 0, feature_base['PTS'] / (2 * (feature_base['FGA'] + 0.44 * feature_base['FTA'])), np.nan)
feature_base['fg3a_rate'] = np.where(feature_base['FGA'] > 0, feature_base['FG3A'] / feature_base['FGA'], np.nan)
feature_base['fta_rate'] = np.where(feature_base['FGA'] > 0, feature_base['FTA'] / feature_base['FGA'], np.nan)
feature_base['ast_tov_ratio'] = feature_base['AST'] / feature_base['TOV'].replace(0, np.nan)
feature_base['ast_tov_ratio'] = feature_base['ast_tov_ratio'].fillna(feature_base['AST'])
feature_base['dreb_pct_est'] = feature_base['DREB'] / (feature_base['DREB'] + feature_base['opp_OREB']).replace(0, np.nan)
feature_base['oreb_pct_est'] = feature_base['OREB'] / (feature_base['OREB'] + feature_base['opp_DREB']).replace(0, np.nan)

historical_signal_preview = feature_base[
    ['TEAM_ABBREVIATION', 'GAME_DATE', 'PTS', 'pts_allowed', 'point_margin', 'off_rating_est', 'def_rating_est', 'net_rating_est', 'ts_pct', 'fg3a_rate']
].head()

historical_signal_preview

,TEAM_ABBREVIATION,GAME_DATE,PTS,pts_allowed,point_margin,off_rating_est,def_rating_est,net_rating_est,ts_pct,fg3a_rate
1,SEA,2001-04-18,105,67.0000,38.0000,119.8630,78.0522,41.8108,0.5731,0.1529
3,DAL,2001-04-18,120,100.0000,20.0000,111.3173,93.9496,17.3676,0.6012,0.1868
5,VAN,2001-04-18,95,81.0000,14.0000,107.6609,84.8701,22.7908,0.4692,0.0978
7,MIA,2001-04-18,103,91.0000,12.0000,109.4348,96.8910,12.5438,0.5196,0.1124
9,DEN,2001-04-18,110,100.0000,10.0000,106.2597,99.4431,6.8165,0.4932,0.1944


## 6. Schedule, Rest, and Streak Features

Rest, back-to-backs, home/away status, games already played, and current streak are known before tip-off. We compute these within each team-season to avoid carrying an offseason as normal rest.

In [6]:
feature_base = feature_base.sort_values(['TEAM_ID', 'GAME_DATE', 'GAME_ID']).reset_index(drop=True)
team_season_group = feature_base.groupby(['TEAM_ID', 'SEASON_YEAR'], sort=False)
team_group = feature_base.groupby('TEAM_ID', sort=False)

feature_base['team_game_number_season'] = team_season_group.cumcount() + 1
feature_base['games_played_before'] = team_season_group.cumcount().astype('int16')
feature_base['prev_game_date'] = team_group['GAME_DATE'].shift(1)
feature_base['prev_game_season'] = team_group['SEASON_YEAR'].shift(1)
feature_base['season_opener'] = (feature_base['games_played_before'] == 0).astype('int8')
feature_base['days_since_last_game'] = (feature_base['GAME_DATE'] - feature_base['prev_game_date']).dt.days
feature_base.loc[feature_base['prev_game_season'] != feature_base['SEASON_YEAR'], 'days_since_last_game'] = np.nan
feature_base['rest_days'] = (feature_base['days_since_last_game'] - 1).clip(lower=0)
feature_base['rest_days_capped'] = feature_base['rest_days'].clip(upper=5)
feature_base['is_back_to_back'] = ((feature_base['rest_days'] == 0) & (feature_base['season_opener'] == 0)).astype('int8')
feature_base['is_3plus_days_rest'] = ((feature_base['rest_days'] >= 3) & (feature_base['season_opener'] == 0)).astype('int8')
feature_base['season_progress'] = feature_base['games_played_before'] / 82

# Fill opener rest with a neutral, explicitly flagged value so model code can run without special casing.
feature_base['days_since_last_game_filled'] = feature_base['days_since_last_game'].fillna(7)
feature_base['rest_days_filled'] = feature_base['rest_days'].fillna(5)
feature_base['rest_days_capped_filled'] = feature_base['rest_days_capped'].fillna(5)

def streak_entering_game(outcomes):
    streaks = []
    current_streak = 0
    for outcome in outcomes.astype(int):
        streaks.append(current_streak)
        if outcome == 1:
            current_streak = current_streak + 1 if current_streak > 0 else 1
        else:
            current_streak = current_streak - 1 if current_streak < 0 else -1
    return pd.Series(streaks, index=outcomes.index)

feature_base['streak_entering_game'] = (
    feature_base
    .groupby(['TEAM_ID', 'SEASON_YEAR'], group_keys=False)['is_win']
    .apply(streak_entering_game)
    .astype('int16')
)

feature_base[['TEAM_ABBREVIATION', 'SEASON_YEAR', 'GAME_DATE', 'games_played_before', 'rest_days_filled', 'is_back_to_back', 'streak_entering_game']].head(12)

,TEAM_ABBREVIATION,SEASON_YEAR,GAME_DATE,games_played_before,rest_days_filled,is_back_to_back,streak_entering_game
0,ATL,2000-01,2000-10-31,0,5.0000,0,0
1,ATL,2000-01,2000-11-02,1,1.0000,0,-1
2,ATL,2000-01,2000-11-04,2,1.0000,0,-2
3,ATL,2000-01,2000-11-06,3,1.0000,0,-3
4,ATL,2000-01,2000-11-07,4,0.0000,1,-4
5,ATL,2000-01,2000-11-09,5,1.0000,0,-5
6,ATL,2000-01,2000-11-10,6,0.0000,1,-6
7,ATL,2000-01,2000-11-14,7,3.0000,0,-7
8,ATL,2000-01,2000-11-15,8,0.0000,1,1
9,ATL,2000-01,2000-11-17,9,1.0000,0,-1


## 7. Rolling and Season-to-Date Team Form

All rolling features are shifted by one game inside each team-season. This turns descriptive box-score patterns from the EDA into legal pre-game predictors. Windows of 5, 10, and 20 games capture short-term form, medium-term stability, and broader team quality.

In [7]:
rolling_source_map = {
    'is_win': 'win_pct',
    'point_margin': 'net_margin',
    'PTS': 'pts_scored',
    'pts_allowed': 'pts_allowed',
    'off_rating_est': 'off_rating',
    'def_rating_est': 'def_rating',
    'net_rating_est': 'net_rating',
    'efg_pct': 'efg_pct',
    'ts_pct': 'ts_pct',
    'FG_PCT': 'fg_pct',
    'FG3_PCT': 'fg3_pct',
    'FT_PCT': 'ft_pct',
    'fg3a_rate': 'fg3a_rate',
    'fta_rate': 'fta_rate',
    'REB': 'reb',
    'OREB': 'oreb',
    'DREB': 'dreb',
    'AST': 'ast',
    'TOV': 'tov',
    'STL': 'stl',
    'BLK': 'blk',
    'PF': 'pf',
    'ast_tov_ratio': 'ast_tov_ratio',
    'dreb_pct_est': 'dreb_pct',
    'oreb_pct_est': 'oreb_pct',
}

rolling_windows = [5, 10, 20]
engineered = feature_base.copy()
team_season_group = engineered.groupby(['TEAM_ID', 'SEASON_YEAR'], sort=False)
rolling_feature_columns = []
season_to_date_columns = []

for source_column, feature_stem in rolling_source_map.items():
    for window in rolling_windows:
        output_column = f'{feature_stem}_roll{window}'
        engineered[output_column] = team_season_group[source_column].transform(
            lambda series, window=window: series.shift(1).rolling(window=window, min_periods=1).mean()
        )
        rolling_feature_columns.append(output_column)

season_to_date_source_map = {
    'is_win': 'win_pct',
    'point_margin': 'net_margin',
    'PTS': 'pts_scored',
    'pts_allowed': 'pts_allowed',
    'off_rating_est': 'off_rating',
    'def_rating_est': 'def_rating',
    'net_rating_est': 'net_rating',
    'ts_pct': 'ts_pct',
    'TOV': 'tov',
}

for source_column, feature_stem in season_to_date_source_map.items():
    output_column = f'{feature_stem}_season_to_date'
    engineered[output_column] = team_season_group[source_column].transform(
        lambda series: series.shift(1).expanding(min_periods=1).mean()
    )
    season_to_date_columns.append(output_column)

engineered[['TEAM_ABBREVIATION', 'SEASON_YEAR', 'GAME_DATE', 'games_played_before', 'win_pct_roll5', 'net_margin_roll5', 'net_rating_roll10', 'win_pct_season_to_date']].head(12)

,TEAM_ABBREVIATION,SEASON_YEAR,GAME_DATE,games_played_before,win_pct_roll5,net_margin_roll5,net_rating_roll10,win_pct_season_to_date
0,ATL,2000-01,2000-10-31,0,NaN,NaN,NaN,NaN
1,ATL,2000-01,2000-11-02,1,0.0000,-24.0000,-29.4050,0.0000
2,ATL,2000-01,2000-11-04,2,0.0000,-24.5000,-28.5828,0.0000
3,ATL,2000-01,2000-11-06,3,0.0000,-17.3333,-20.6142,0.0000
4,ATL,2000-01,2000-11-07,4,0.0000,-15.5000,-17.9180,0.0000
5,ATL,2000-01,2000-11-09,5,0.0000,-14.2000,-16.4694,0.0000
6,ATL,2000-01,2000-11-10,6,0.0000,-11.2000,-15.0857,0.0000
7,ATL,2000-01,2000-11-14,7,0.0000,-8.0000,-13.9889,0.0000
8,ATL,2000-01,2000-11-15,8,0.2000,-7.0000,-12.6036,0.1250
9,ATL,2000-01,2000-11-17,9,0.2000,-7.0000,-13.2493,0.1111


## 8. Prior-Season Advanced Team Priors

The raw `team_stats` files contain final season ratings. For games inside a season, using that season's final value would leak future games. We therefore join only the previous season's final advanced metrics as priors. Expansion or missing prior rows are filled with the previous season league average; the first collected season remains unavailable and is removed from the stricter model-ready output.

In [8]:
advanced_prior_columns = [
    'W_PCT', 'OFF_RATING', 'DEF_RATING', 'NET_RATING', 'AST_PCT', 'AST_TO', 'AST_RATIO',
    'OREB_PCT', 'DREB_PCT', 'REB_PCT', 'TM_TOV_PCT', 'EFG_PCT', 'TS_PCT', 'PACE', 'PIE'
]

prior_team_stats = team_stats_clean[['TEAM_ID', 'SEASON', 'season_start_year'] + advanced_prior_columns].copy()
prior_team_stats['season_start_year'] = prior_team_stats['season_start_year'] + 1
prior_team_stats = prior_team_stats.rename(
    columns={column: f'prior_season_{column.lower()}' for column in advanced_prior_columns}
).drop(columns=['SEASON'])

prior_league_average = (
    team_stats_clean[['season_start_year'] + advanced_prior_columns]
    .groupby('season_start_year', as_index=False)
    .mean(numeric_only=True)
)
prior_league_average['season_start_year'] = prior_league_average['season_start_year'] + 1
prior_league_average = prior_league_average.rename(
    columns={column: f'prior_league_avg_{column.lower()}' for column in advanced_prior_columns}
)

engineered = engineered.merge(prior_team_stats, on=['TEAM_ID', 'season_start_year'], how='left')
engineered = engineered.merge(prior_league_average, on='season_start_year', how='left')

prior_feature_columns = [f'prior_season_{column.lower()}' for column in advanced_prior_columns]
prior_league_columns = [f'prior_league_avg_{column.lower()}' for column in advanced_prior_columns]

for team_column, league_column in zip(prior_feature_columns, prior_league_columns):
    engineered[team_column] = engineered[team_column].fillna(engineered[league_column])

engineered[['TEAM_ABBREVIATION', 'SEASON_YEAR', 'prior_season_w_pct', 'prior_season_net_rating', 'prior_season_off_rating', 'prior_season_def_rating']].head()

,TEAM_ABBREVIATION,SEASON_YEAR,prior_season_w_pct,prior_season_net_rating,prior_season_off_rating,prior_season_def_rating
0,ATL,2000-01,NaN,NaN,NaN,NaN
1,ATL,2000-01,NaN,NaN,NaN,NaN
2,ATL,2000-01,NaN,NaN,NaN,NaN
3,ATL,2000-01,NaN,NaN,NaN,NaN
4,ATL,2000-01,NaN,NaN,NaN,NaN


## 9. Join Opponent Pre-Game Features

A game prediction depends on both teams. We join the opponent's shifted features onto each team row, then create matchup-difference features such as team rolling net rating minus opponent rolling net rating.

In [9]:
context_feature_columns = [
    'games_played_before', 'season_opener', 'days_since_last_game_filled', 'rest_days_filled',
    'rest_days_capped_filled', 'is_back_to_back', 'is_3plus_days_rest', 'season_progress',
    'streak_entering_game',
]

team_feature_columns = context_feature_columns + rolling_feature_columns + season_to_date_columns + prior_feature_columns

opponent_feature_frame = engineered[['GAME_ID', 'TEAM_ID'] + team_feature_columns].rename(columns={'TEAM_ID': 'opponent_team_id'})
opponent_feature_frame = opponent_feature_frame.rename(
    columns={column: f'opp_{column}' for column in team_feature_columns}
)

with_opponent_features = engineered.merge(opponent_feature_frame, on=['GAME_ID', 'opponent_team_id'], how='left')

matchup_base_columns = [
    'win_pct_roll5', 'win_pct_roll10', 'win_pct_roll20',
    'net_margin_roll5', 'net_margin_roll10', 'net_margin_roll20',
    'net_rating_roll5', 'net_rating_roll10', 'net_rating_roll20',
    'off_rating_roll10', 'def_rating_roll10', 'ts_pct_roll10', 'tov_roll10',
    'win_pct_season_to_date', 'net_rating_season_to_date', 'prior_season_w_pct',
    'prior_season_net_rating', 'prior_season_off_rating', 'prior_season_def_rating',
    'rest_days_capped_filled', 'streak_entering_game',
]

matchup_difference_columns = []
for column in matchup_base_columns:
    diff_column = f'team_minus_opp_{column}'
    with_opponent_features[diff_column] = with_opponent_features[column] - with_opponent_features[f'opp_{column}']
    matchup_difference_columns.append(diff_column)

# Cross-strength features compare a team's offense to the opponent's defense and vice versa.
with_opponent_features['roll10_offense_vs_opp_defense'] = with_opponent_features['off_rating_roll10'] - with_opponent_features['opp_def_rating_roll10']
with_opponent_features['roll10_defense_vs_opp_offense'] = with_opponent_features['opp_off_rating_roll10'] - with_opponent_features['def_rating_roll10']
with_opponent_features['prior_offense_vs_opp_defense'] = with_opponent_features['prior_season_off_rating'] - with_opponent_features['opp_prior_season_def_rating']
with_opponent_features['prior_defense_vs_opp_offense'] = with_opponent_features['opp_prior_season_off_rating'] - with_opponent_features['prior_season_def_rating']

cross_strength_columns = [
    'roll10_offense_vs_opp_defense', 'roll10_defense_vs_opp_offense',
    'prior_offense_vs_opp_defense', 'prior_defense_vs_opp_offense',
]

assert with_opponent_features.shape[0] == engineered.shape[0]
assert not with_opponent_features[[f'opp_{column}' for column in context_feature_columns]].isna().all(axis=1).any()

with_opponent_features[['TEAM_ABBREVIATION', 'opponent_abbr', 'GAME_DATE', 'win_pct_roll10', 'opp_win_pct_roll10', 'team_minus_opp_win_pct_roll10', 'roll10_offense_vs_opp_defense']].head(10)

,TEAM_ABBREVIATION,opponent_abbr,GAME_DATE,win_pct_roll10,opp_win_pct_roll10,team_minus_opp_win_pct_roll10,roll10_offense_vs_opp_defense
0,ATL,CHH,2000-10-31,NaN,NaN,NaN,NaN
1,ATL,NYK,2000-11-02,0.0000,0.0000,0.0000,-29.6461
2,ATL,ORL,2000-11-04,0.0000,0.3333,-0.3333,-6.6716
3,ATL,VAN,2000-11-06,0.0000,0.6667,-0.6667,-4.3457
4,ATL,POR,2000-11-07,0.0000,0.2500,-0.2500,-10.7519
5,ATL,PHX,2000-11-09,0.0000,0.8000,-0.8000,-3.5060
6,ATL,LAC,2000-11-10,0.0000,0.2000,-0.2000,-9.2476
7,ATL,POR,2000-11-14,0.0000,0.6250,-0.6250,-4.7034
8,ATL,MIL,2000-11-15,0.1250,0.1667,-0.0417,-14.4810
9,ATL,BOS,2000-11-17,0.1111,0.4286,-0.3175,-3.8206


## 10. Final Feature Tables

`team_game_features.csv` keeps every team-game row and may contain missing values for season openers and the first collected season. `modeling_dataset.csv` is stricter: it removes rows where either team has no current-season history and removes the first collected season because no prior-season advanced metrics are available from our raw data.

In [10]:
identifier_columns = [
    'SEASON_YEAR', 'season_start_year', 'season_order', 'GAME_ID', 'GAME_DATE', 'game_year', 'game_month',
    'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'MATCHUP', 'opponent_team_id', 'opponent_abbr',
    'opponent_team_name', 'is_home', 'is_road', 'is_neutral_or_unresolved',
]
target_columns = ['WL', 'is_win']

opponent_team_feature_columns = [f'opp_{column}' for column in team_feature_columns]
final_feature_columns = team_feature_columns + opponent_team_feature_columns + matchup_difference_columns + cross_strength_columns

team_game_features = with_opponent_features[identifier_columns + target_columns + final_feature_columns].copy()
team_game_features = team_game_features.sort_values(['GAME_DATE', 'GAME_ID', 'TEAM_ID']).reset_index(drop=True)

first_season_start_year = int(team_game_features['season_start_year'].min())
modeling_dataset = team_game_features.loc[
    (team_game_features['season_start_year'] > first_season_start_year)
    & (team_game_features['games_played_before'] > 0)
    & (team_game_features['opp_games_played_before'] > 0)
].copy()

numeric_feature_columns = modeling_dataset[final_feature_columns].select_dtypes(include='number').columns.tolist()
remaining_missing = modeling_dataset[numeric_feature_columns].isna().sum().loc[lambda counts: counts.gt(0)]

team_game_features.to_csv(FEATURES_PATH, index=False)
modeling_dataset.to_csv(MODEL_READY_PATH, index=False)

pd.DataFrame(
    [
        {'output': str(FEATURES_PATH), 'rows': team_game_features.shape[0], 'columns': team_game_features.shape[1], 'missing_numeric_feature_values': int(team_game_features[final_feature_columns].select_dtypes(include='number').isna().sum().sum())},
        {'output': str(MODEL_READY_PATH), 'rows': modeling_dataset.shape[0], 'columns': modeling_dataset.shape[1], 'missing_numeric_feature_values': int(remaining_missing.sum())},
    ]
)

,output,rows,columns,missing_numeric_feature_values
0,data/processed/team_game_features.csv,60048,260,225318
1,data/processed/modeling_dataset.csv,56856,260,0


## 11. Validation and Leakage Audit

These checks are intentionally strict. The final feature tables should not contain current-game box-score columns such as `PTS`, `PLUS_MINUS`, `FG_PCT`, or `AST` as direct predictors. Those ideas appear only inside shifted rolling, season-to-date, prior-season, opponent, or difference features.

In [11]:
leaky_direct_columns = {
    'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT',
    'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS',
    'point_margin', 'pts_allowed', 'off_rating_est', 'def_rating_est', 'net_rating_est',
}

leaky_columns_found = sorted(leaky_direct_columns.intersection(team_game_features.columns))
assert not leaky_columns_found, f'Direct current-game leakage columns found: {leaky_columns_found}'
assert not team_game_features.duplicated(['GAME_ID', 'TEAM_ID']).any()
assert team_game_features.groupby('GAME_ID').size().eq(2).all()
assert modeling_dataset['is_win'].value_counts().nunique() == 1, 'Model-ready data should retain paired win/loss balance.'
assert remaining_missing.empty, f'Model-ready numeric features still have missing values: {remaining_missing.to_dict()}'

validation_report = pd.DataFrame(
    [
        {'check': 'No direct current-game box score predictors', 'status': 'passed'},
        {'check': 'One row per team per game', 'status': 'passed'},
        {'check': 'Two rows per game in full feature table', 'status': 'passed'},
        {'check': 'Model-ready target is balanced', 'status': 'passed'},
        {'check': 'No missing numeric model-ready features', 'status': 'passed'},
    ]
)

validation_report

,check,status
0,No direct current-game box score predictors,passed
1,One row per team per game,passed
2,Two rows per game in full feature table,passed
3,Model-ready target is balanced,passed
4,No missing numeric model-ready features,passed


## 12. Feature Dictionary

This file gives notebook 04 a compact map of identifiers, target fields, and engineered predictors.

In [12]:
def feature_role(column):
    if column in identifier_columns:
        return 'identifier/context'
    if column in target_columns:
        return 'target'
    if column.startswith('opp_'):
        return 'opponent pre-game feature'
    if column.startswith('team_minus_opp_'):
        return 'matchup difference feature'
    if column in cross_strength_columns:
        return 'matchup cross-strength feature'
    return 'team pre-game feature'

feature_dictionary = pd.DataFrame(
    {
        'column': team_game_features.columns,
        'role': [feature_role(column) for column in team_game_features.columns],
        'missing_in_full_feature_table': [int(team_game_features[column].isna().sum()) for column in team_game_features.columns],
        'dtype': [str(team_game_features[column].dtype) for column in team_game_features.columns],
    }
)

feature_dictionary.to_csv(DICTIONARY_PATH, index=False)

feature_dictionary.groupby('role').size().to_frame('columns')

,columns
role,
identifier/context,17
matchup cross-strength feature,4
matchup difference feature,21
opponent pre-game feature,108
target,2
team pre-game feature,108


## Final Notes for Notebook 04

- Use `data/processed/modeling_dataset.csv` as the default modeling input.
- Keep time-aware validation: train on earlier seasons and test on later seasons.
- Do not randomly split team-game rows from the same season without a temporal strategy.
- Start with interpretable baselines using the matchup difference features, then add tree models.
- `NET_RATING` is included only as prior-season, shifted rolling, season-to-date, opponent, and difference versions. Early-season rows use shorter rolling history, and season openers are excluded from the model-ready file.